# Parcel-Facility Join

In [89]:
import geopandas as gpd 
import pandas as pd
import numpy as np

# load parcels
parcels = gpd.read_file("C:/Users/Bryce/Main/CMU/Fall 2026/Python II/Final Project/assessor_parcels_analytic.geojson")

# load facilities
facilities_gdf = (
    gpd.read_file("major_fac.geojson")
)

In [99]:
facilities_gdf=facilities_gdf.reset_index().set_index("trifd")
print(facilities_gdf[["name", "air_lbs"]].sort_values("air_lbs"))  # check for zeros before logging

dist = pd.DataFrame(
    {fid: parcels.geometry.distance(geom) / 5280 for fid, geom in facilities_gdf.geometry.items()},
    index=parcels.index,
)  # miles, one column per facility

w_log = np.log10(facilities_gdf["air_lbs"] + 1)
w_log = w_log / w_log.mean()  # mean weight of 1, same scale as unweighted

for lam in [0.5, 1, 2]:
    decay = np.exp(-dist / lam)
    parcels[f"exp_unw_l{lam}"] = decay.sum(axis=1)
    parcels[f"exp_log_l{lam}"] = (decay * w_log).sum(axis=1)  # aligns on trifd

                                                              name    air_lbs
trifd                                                                        
15201MCCNW10948                     MCCONWAY & TORLEY LLC PLT 1801     224.52
1511WTHRML5NRTH  WABTEC COMPONENTS LLC FORMERLY THERMAL TRANSFE...     260.00
15017CYTMPMAYER           UNIVERSAL STAINLESS & ALLOY PRODUCTS INC     488.00
15014LLGHNRIVER              ATI FLAT ROLLED PRODUCTS HOLDINGS LLC    1419.00
15225NVLLC2800N                                NEVILLE CHEMICAL CO   14392.00
15144PPGND125CO               PPG INDUSTRIESINC-SPRINGDALE COMPLEX   22186.09
15104MSDVT13BRA                              TMS INTERNATIONAL LLC   27877.98
15088HRCLSSTATE                      SYNTHOMER JEFFERSON HILLS LLC   50394.00
15034SSRVNPOBOX                 USS MON VALLEY WORKS - IRVIN PLANT   73871.60
15104SSDGRBRADD         USS MON VALLEY WORKS - EDGAR THOMSON PLANT   96883.69
15025SSCLR400ST                                 USS-CLAIRTON PLA

In [61]:
# # inspect results with density plots of each of the decay variables
# import matplotlib.pyplot as plt
# for col in parcels.columns:
#     if col.startswith("exp_"):
#         plt.figure()
#         parcels[col].hist(bins=50, density=True)
#         parcels[col].plot(kind='kde')
#         plt.title(col)
#         plt.show()

In [62]:
parcels.columns

Index(['parid', 'propertycity', 'class', 'classdesc', 'ownerdesc', 'grade',
       'saledate', 'saleprice', 'countybuilding', 'countyland', 'countytotal',
       'fairmarketbuilding', 'fairmarketland', 'fairmarkettotal', 'taxyear',
       'sale_day', 'sale_month', 'sale_year', '_id', 'map_block_lot',
       'municode', 'calc_acreage', 'comments', 'notes', 'pseudono',
       'shape_length', 'index_right', 'name', 'type', 'region', 'sqmi', 'eoc',
       'lat', 'lon', 'geometry', 'exp_unw_l0.5', 'exp_log_l0.5', 'exp_unw_l1',
       'exp_log_l1', 'exp_unw_l2', 'exp_log_l2'],
      dtype='str')

In [63]:
# download census tract shapefiles

import pygris
from dotenv import load_dotenv
import os
import census as c
from us import states

load_dotenv()
CENSUS_API_KEY = "e42f6b0ecfc65388715dc2caf3cfa9ae6b5a3c54"

if CENSUS_API_KEY is None:
    print("Error: API key not found. Make sure .env file exists with OPENWEATHER_API_KEY set.")
else:
    print("API key loaded successfully!")

# alleg = pygris.counties(states.PA.fips).query("NAME == 'Allegheny'").to_crs("EPSG:2272")
allg_tracts = pygris.tracts(states.PA.fips).query("COUNTYFP == '003'").to_crs("EPSG:2272")

API key loaded successfully!
Using the default year of 2024


In [ ]:
# read the full ACS DataFrame from the geojson file
from pathlib import Path

output_dir = Path(r"C:\Users\Bryce\Main\CMU\Fall 2026\Python II\Final Project")
output_dir.mkdir(parents=True, exist_ok=True)

acs_5_gdf = gpd.read_file(output_dir / "full_acs_df.geojson").rename(columns={"name": "cnty_name"})

# calculate pop white share of total population
acs_5_gdf["pop_white_share"] = acs_5_gdf["pop_white"] / acs_5_gdf["pop_total"]

# calculate college degree share of total population
acs_5_gdf["college_degree_share"] = (acs_5_gdf["Associate's degree"] + acs_5_gdf["Bachelor's degree"] + acs_5_gdf["Master's degree"] + acs_5_gdf["Doctorate degree"] + acs_5_gdf["Professional school degree"]) / acs_5_gdf["pop_total"]

# calculate vacancy rate
acs_5_gdf["vacancy_rate"] = acs_5_gdf["vac_housing"] / acs_5_gdf["tot_housing"]


# subset acs_5_gdf to pop_total, tractfips
acs_5_gdf_subset = acs_5_gdf[["geoid",
                              "pop_total", 
                              "statefips", 
                              "countyfips", 
                              "tractfips",
                              "cnty_name",
                              "hhinc",
                              "home_val",
                              "pop_white_share",
                              "college_degree_share",
                              "vacancy_rate",
                              "geometry"
                              ]].set_crs(crs="EPSG:2272", allow_override=True).copy()

In [65]:
acs_5_gdf.columns

Index(['geoid', 'pop_total', 'statefips', 'countyfips', 'tractfips',
       'cnty_name', 'pop_white', 'pop_black', 'pop_asian', 'pop_hispan',
       'hhinc', 'home_val', 'tot_housing', 'occ_housing', 'vac_housing',
       'occ_by_tenure', 'owners', 'renters',
       'Population age 25 and over (education universe)',
       'No schooling completed', 'Nursery school', 'Kindergarten', '1st grade',
       '2nd grade', '3rd grade', '4th grade', '5th grade', '6th grade',
       '7th grade', '8th grade', '9th grade', '10th grade', '11th grade',
       '12th grade, no diploma', 'Regular high school diploma',
       'GED or alternative credential', 'Some college, less than 1 year',
       'Some college, 1 or more years, no degree', 'Associate's degree',
       'Bachelor's degree', 'Master's degree', 'Professional school degree',
       'Doctorate degree', 'geometry', 'pop_white_share',
       'college_degree_share', 'vacancy_rate'],
      dtype='str')

In [101]:
# convert parcels to centroid by dropping geometry column and creating new geom column from lat lon 
parcels_centroid = parcels.drop(columns="geometry")
parcels_centroid["geom"] = gpd.points_from_xy(parcels_centroid["lon"], parcels_centroid["lat"])
parcels_centroid = gpd.GeoDataFrame(parcels_centroid, geometry="geom", crs="EPSG:2272")

parcels_centroid = parcels_centroid.drop(columns=["index_right"], errors="ignore")
acs_5_gdf_subset = acs_5_gdf_subset.drop(columns=["index_right"], errors="ignore")

# join parcels with census tracts 
parcels_with_tracts = gpd.sjoin(
    parcels_centroid, 
    acs_5_gdf_subset, 
    how="left", predicate="intersects")

# take just parcel id and tract id
parcels_with_tracts = parcels_with_tracts[[
    "parid", 
    "geoid",
    "pop_total", 
    "statefips", 
    "countyfips", 
    "tractfips",
    "cnty_name",
    "hhinc",
    "home_val",
    "pop_white_share",
    "college_degree_share",
    "vacancy_rate"]]

# rejoin tract info to original parcels file 
parcels = parcels.merge(parcels_with_tracts, on="parid", how="left")

# save new parcels 
# output_dir = Path(r"C:\Users\Bryce\Main\CMU\Fall 2026\Python II\Final Project")
# output_dir.mkdir(parents=True, exist_ok=True)

parcels.to_file(output_dir / "parcels_with_tracts.geojson", driver="GeoJSON")

In [58]:
parcels.columns

Index(['parid', 'propertycity', 'class', 'classdesc', 'ownerdesc', 'grade',
       'saledate', 'saleprice', 'countybuilding', 'countyland', 'countytotal',
       'fairmarketbuilding', 'fairmarketland', 'fairmarkettotal', 'taxyear',
       'sale_day', 'sale_month', 'sale_year', '_id', 'map_block_lot',
       'municode', 'calc_acreage', 'comments', 'notes', 'pseudono',
       'shape_length', 'index_right', 'name', 'type', 'region', 'sqmi', 'eoc',
       'lat', 'lon', 'geometry', 'exp_unw_l0.5', 'exp_log_l0.5', 'exp_unw_l1',
       'exp_log_l1', 'exp_unw_l2', 'exp_log_l2'],
      dtype='str')

In [ ]:
acs_5_gdf[["geoid","tot_housing","occ_by_tenure","occ_housing","owners","renters","vac_housing"]]

In [ ]:
# merge parcels with acs_5_gdf_subset
# parcels_with_acs = parcels.merge(acs_5_gdf_subset, on="geoid", how="left")

In [77]:
parcels.columns

Index(['parid', 'propertycity', 'class', 'classdesc', 'ownerdesc', 'grade',
       'saledate', 'saleprice', 'countybuilding', 'countyland', 'countytotal',
       'fairmarketbuilding', 'fairmarketland', 'fairmarkettotal', 'taxyear',
       'sale_day', 'sale_month', 'sale_year', '_id', 'map_block_lot',
       'municode', 'calc_acreage', 'comments', 'notes', 'pseudono',
       'shape_length', 'index_right', 'name', 'type', 'region', 'sqmi', 'eoc',
       'lat', 'lon', 'geometry', 'exp_unw_l0.5', 'exp_log_l0.5', 'exp_unw_l1',
       'exp_log_l1', 'exp_unw_l2', 'exp_log_l2', 'geoid', 'pop_total',
       'statefips', 'countyfips', 'tractfips', 'cnty_name', 'hhinc',
       'home_val', 'pop_white_share', 'college_degree_share', 'vacancy_rate'],
      dtype='str')

In [ ]:
# write parcels to geojson
# parcels.to_file(output_dir / "90819_analysis_data_full.geojson", driver="GeoJSON")

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np
import statsmodels.api as sm

# filter acs_5_gdf for rows with a population and non-NA vacancy rate
feature_cols = ["exp_log_l1","hhinc", "home_val", "pop_white_share", "college_degree_share"]

parcels_subset = parcels.loc[(parcels["pop_total"] > 0)].dropna(subset=feature_cols).copy()
# keep only parcels where classdesc is residential and sale_year is 2024
parcels_subset = parcels_subset.query("classdesc.str.contains('residential', case=False) and sale_year == 2024")

# write a regression analysis in sklearn
X = parcels_subset[feature_cols].apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(parcels_subset["saleprice"], errors="coerce")

valid = X.notna().all(axis=1) & y.notna()
valid &= np.isfinite(X.to_numpy(dtype=float)).all(axis=1)
valid &= np.isfinite(y.to_numpy(dtype=float))

X = X.loc[valid].to_numpy(dtype=float)
y = y.loc[valid].to_numpy(dtype=float)

model_1 = LinearRegression().fit(X, y)
print("Intercept:", model_1.intercept_)
print("Coefficients:", model_1.coef_)
print("R^2 Score:", model_1.score(X, y))

Intercept: 48381.54840821981
Coefficients: [-2.19878540e+04 -1.08125347e+00  9.07806344e-01 -1.36189366e+04
 -8.81107157e+02]
R^2 Score: 0.01138374392990782


In [98]:
parcels_subset.shape

(24139, 52)

In [96]:
# write parcels to geojson
parcels_subset.to_file(output_dir / "90819_analysis_data.geojson", driver="GeoJSON")

In [75]:
from patsy import dmatrices

feature_cols = ["exp_log_l1","hhinc", "home_val", "pop_white_share", "college_degree_share"]
model_data = parcels.loc[parcels["pop_total"] > 0, ["saleprice", *feature_cols]].apply(pd.to_numeric, errors="coerce").dropna()

y, X = dmatrices("saleprice ~ exp_log_l1 + hhinc + home_val + pop_white_share + college_degree_share", data=model_data, return_type="dataframe")
model_2 = sm.OLS(y, X).fit()
print(model_2.summary())

                            OLS Regression Results                            
Dep. Variable:              saleprice   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                  0.011
Method:                 Least Squares   F-statistic:                     1319.
Date:                Fri, 09 Oct 2026   Prob (F-statistic):               0.00
Time:                        18:06:12   Log-Likelihood:            -8.6425e+06
No. Observations:              572556   AIC:                         1.729e+07
Df Residuals:                  572550   BIC:                         1.729e+07
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             4.731e+04 